# Lab 01: How to search documents with embeddings

**CPU | API key: `OPENAI_API_KEY` (embedding calls only, no chat model) | In class**

## Goal

Search a set of documents for a question with an embeddings API, check how often the right paragraph comes back, and
learn what a similarity score cannot tell you. The moment that teaches it: a question that no paragraph answers
scores higher than questions the paragraphs do answer, so no score cutoff can tell the two kinds apart.

## What you will learn

- How to build an embeddings client, embed a text and read the vector that comes back.
- Why cosine similarity is a plain dot product for the unit-length vectors the API returns.
- How to embed 272 chunks in batches and keep every row tied to its chunk ID.
- How to rank every chunk for 30 questions with one matrix product and count known-source hit@3.
- How to test, on questions labelled before the search, whether a score cutoff tells answerable questions from
  unanswerable ones.
- How to search the corpus with `shared/search.py`, which keeps these steps for the labs after this one.

**In class:** Parts A, B, C and E. Parts D, F and G are self-study. Run Part D's cells in class without stopping on
them, because Part E ranks the vectors Part D builds.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-24
OPENAI_API_KEY    found in .env


This lab reads two of the printed lines, `embed_model` and `embed_endpoint`: every score below comes from that model at that address. It calls no chat model, and all its calls together cost well under one US cent. Scores can move in the third decimal from one run to the next, so the text gives them to two decimals: compare the order of results with the text, not the last digit.

---

## Part A: Embed one question

Part A makes the call that every search in these labs starts with: a text goes in, a list of numbers comes out.

An **embedding** is a list of numbers, a **vector**, that a model computes for a text so that texts with similar
meanings get vectors pointing in similar directions. You get one through a **client**, the OpenAI SDK object that
sends requests. A client needs the endpoint's address and your key: `embed_url(settings)` returns the address the
settings name, and Config put the key in the environment. Build one by hand:

In [3]:
import os
from openai import OpenAI
from shared.settings import embed_url

client = OpenAI(base_url=embed_url(settings), api_key=os.environ["OPENAI_API_KEY"])
print("Embeddings endpoint:", client.base_url)

Embeddings endpoint: https://api.openai.com/v1/


Now send one question. `input` takes a list of texts; the reply holds one result per text in `response.data`, in the
same order, and `usage` says how many tokens you were billed.

In [4]:
question = "In what year was Nikola Tesla born?"
response = client.embeddings.create(model=settings["EMBED_MODEL"], input=[question])
vector = response.data[0].embedding

print("model:", response.model)
print("numbers in the vector:", len(vector))
print("first five:", [round(number, 4) for number in vector[:5]])
print("tokens billed:", response.usage.prompt_tokens)

model: text-embedding-3-small
numbers in the vector: 1536
first five: [-0.032, -0.0686, -0.0369, -0.0109, 0.0071]
tokens billed: 8


The model returns 1,536 numbers for this question, and as many for any other text: the length is fixed by the
model. One number means nothing on its own; Part B compares two vectors. The question cost 8 tokens, and
embedding models bill input only.

Every lab needs this client, so `embed_client()` in `shared/settings.py` builds it from the settings. Read it, then use it:

In [5]:
import inspect
from shared.settings import embed_client

print(inspect.getsource(embed_client))
client = embed_client(settings)

def embed_client(settings: dict | None = None):
    """An OpenAI client for EMBED_MODEL (Embeddings). EMBED_API_KEY is used only with EMBED_BASE_URL."""
    from openai import OpenAI
    settings = settings or load_settings()
    key = os.environ.get("OPENAI_API_KEY")
    if settings.get("EMBED_BASE_URL") and load_key("EMBED_API_KEY"):
        key = os.environ["EMBED_API_KEY"]
    return OpenAI(base_url=embed_url(settings), api_key=key)



Its last line is your line. The `if` before it matters only when `models.env` sends embeddings to a second provider with its own key; with the shipped settings it does nothing. The cell switched to it, and every lab from here on uses `embed_client(settings)`.

### Checkpoint A

| You should see | Our run |
|---|---|
| Endpoint | `https://api.openai.com/v1/`, or the address you set in `models.env` or `.env` |
| The vector | `text-embedding-3-small`, 1,536 numbers, 8 tokens billed |

---

## Part B: Measure closeness with cosine similarity

Part B gives you the number a search ranks by: how close two vectors point.

**Cosine similarity** compares directions. Multiply two vectors' matching numbers and add up the products (the **dot
product**, `@` in NumPy), then divide by the two vectors' lengths. The result is 1 for the same direction, 0 for
unrelated directions and -1 for opposite ones. Try it first on vectors of two numbers, where you can check every value:
a question `q` and two documents, `A` and `B`.

In [6]:
import numpy as np

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

q = np.array([1.0, 1.0])
A = np.array([1.0, 1.0])
B = np.array([5.0, 0.0])
print("dot:    A", q @ A, "  B", q @ B)
print("cosine: A", round(cosine(q, A), 4), "  B", round(cosine(q, B), 4))

dot:    A 2.0   B 5.0
cosine: A 1.0   B 0.7071


By the dot product B wins, 5 against 2, only because B is long. By cosine A wins: it points exactly the way `q`
points (1.0), while B is 45 degrees off (0.7071). Dividing by the lengths removes the effect of length.

Now three real sentences, embedded in one call. Before you run the cell, decide which two you expect to be closest.

In [7]:
sentences = [
    "How long does an Atlas reset link remain valid?",
    "When does the Atlas password link stop working?",
    "Atlas invoices are sent monthly.",
]
response = client.embeddings.create(model=settings["EMBED_MODEL"], input=sentences)
vectors = np.array([item.embedding for item in response.data])
print("shape:", vectors.shape)
print("lengths:", np.round(np.linalg.norm(vectors, axis=1), 3))

shape: (3, 1536)
lengths: [1. 1. 1.]


One row per sentence, 1,536 numbers each, and every length is 1 to three decimals. OpenAI returns embeddings
already **normalised** to length 1 ([embeddings guide](https://developers.openai.com/api/docs/guides/embeddings)), so `cosine` divides by 1: for these vectors
the dot product alone is the cosine. Check it on every pair:

In [8]:
for i, j in [(0, 1), (0, 2), (1, 2)]:
    print(f"sentences {i + 1} and {j + 1}: cosine {cosine(vectors[i], vectors[j]):.4f}   dot {vectors[i] @ vectors[j]:.4f}")

sentences 1 and 2: cosine 0.7307   dot 0.7308
sentences 1 and 3: cosine 0.3137   dot 0.3136
sentences 2 and 3: cosine 0.2729   dot 0.2728


Sentences 1 and 2 score about 0.73: they share few words and ask the same thing. The invoice sentence is far from both
(about 0.31 and 0.27). The dot product matches the cosine, so the rest of the lab ranks with `@` alone. A score of
0.73 does not mean "73 percent the same": another model gives other values for the same pair, and
only the order of one model's scores means something.

Not every endpoint normalises. If your lengths are not 1, divide each vector by its length once
(`vectors / np.linalg.norm(vectors, axis=1, keepdims=True)`) before you treat `@` as the cosine.

### Checkpoint B

| You should see | Our run |
|---|---|
| Toy vectors | dot A 2.0, B 5.0; cosine A 1.0, B 0.7071 |
| Real sentences | shape `(3, 1536)`, lengths 1.0; pairs 1–2 about 0.73, 1–3 about 0.31, 2–3 about 0.27 |

Scores from the API can move by a few thousandths between runs; check the order.

---

## Part C: Search the five product policies

Part C turns the score into a search: embed the question, embed every document, score each one and sort. Five short
policies are few enough to read every result.

`data/support-case.json` holds five two-sentence policies for two made-up products, Atlas and Boreal.

In [9]:
import json

DATA = ROOT / "data"
policies = json.loads((DATA / "support-case.json").read_text(encoding="utf-8"))
for policy in policies:
    print(policy["source_id"], policy["text"])

H01 Atlas passwords can be reset by email. Temporary accounts require a support agent.
H02 Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.
H03 Atlas exports reports as CSV files. Only workspace owners can export all reports.
H04 Boreal reset links expire after 10 minutes. Password reset uses a mobile app.
H05 Atlas invoices are sent monthly. Workspace owners can download invoices from Billing.


The question below asks about Atlas, and only H02 answers it. Predict which policy comes second.

In [10]:
response = client.embeddings.create(model=settings["EMBED_MODEL"], input=[policy["text"] for policy in policies])
policy_vectors = np.array([item.embedding for item in response.data])

atlas_question = "How long does an Atlas reset link remain valid?"
response = client.embeddings.create(model=settings["EMBED_MODEL"], input=[atlas_question])
question_vector = np.array(response.data[0].embedding)

policy_scores = policy_vectors @ question_vector        # one score per policy
for i in np.argsort(-policy_scores):                    # row numbers, best score first
    print(f"{policy_scores[i]:.4f}  {policies[i]['source_id']}  {policies[i]['text']}")

0.6970  H02  Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.
0.4882  H04  Boreal reset links expire after 10 minutes. Password reset uses a mobile app.
0.4254  H01  Atlas passwords can be reset by email. Temporary accounts require a support agent.
0.2401  H05  Atlas invoices are sent monthly. Workspace owners can download invoices from Billing.
0.1775  H03  Atlas exports reports as CSV files. Only workspace owners can export all reports.


`policy_vectors @ question_vector` scores all five policies in one product. `np.argsort(-scores)` returns the row
numbers from the highest score down (NumPy sorts smallest first; the minus sign turns that around).

H02 comes first (about 0.70) and answers: 30 minutes. Second is H04 (about 0.49), the **Boreal** reset policy, above
H01, which is about Atlas. H04 shares the question's topic and words (*reset links expire*) and answers for the wrong
product: its 10 minutes would be a wrong answer about Atlas. Similarity ranks candidates by topic and wording; only the
passage text shows whether one supports an answer. That is why every search in these labs returns each chunk's text
and ID with its score.

### Checkpoint C

| You should see | Our run |
|---|---|
| Ranking | H02 about 0.70, H04 0.49, H01 0.43, H05 0.24, H03 0.18, in that order |

---

## Part D: Embed the corpus in batches (self-study)

Part D builds what the rest of the lab searches: one vector for each of the 272 chunks, each row tied to its chunk ID.

`data/squad-chunks.json` holds 120 Wikipedia paragraphs from **SQuAD**, the Stanford Question Answering Dataset, a well-known public benchmark of questions people wrote about Wikipedia articles ([Rajpurkar et al., 2016](https://arxiv.org/abs/1606.05250)). The paragraphs are split into 272 **chunks**, the pieces a search ranks and
returns. Each chunk keeps the ID of the paragraph it came from (`source_id`) and has its own (`chunk_id`). We cut them once,
with a simple rule (96 tokens per chunk, a new chunk every 80 tokens), and every lab searches these same chunks, so
results can be compared from lab to lab. SQuAD's own file keeps each paragraph together with its questions; here the
questions are in a separate file that the search never sees. Lab 03 shows how the chunks were cut and tests better ways.

In [11]:
chunks = json.loads((DATA / "squad-chunks.json").read_text(encoding="utf-8"))
print(len(chunks), "chunks")
print({name: value for name, value in chunks[0].items() if name != "text"})
print(chunks[0]["text"][:120], "...")

272 chunks
{'chunk_id': 'S00-P00-C00', 'source_id': 'S00-P00', 'title': 'Super_Bowl_50', 'start': 0, 'end': 486, 'tokens': 98}
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 ...


Send lists of texts, not one text per request. OpenAI takes up to 2,048 texts and 300,000 tokens in one request
([API reference](https://developers.openai.com/api/reference/resources/embeddings/methods/create)), so these 272 chunks would fit in one; other servers set lower limits and a real corpus is
larger, so the loop sends batches of 100. Build the texts and the IDs from the same list of chunks, in the same order:
row `i` of the matrix then belongs to `chunk_ids[i]`.

In [12]:
texts = [chunk["text"] for chunk in chunks]
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

rows = []
for start in range(0, len(texts), 100):
    response = client.embeddings.create(model=settings["EMBED_MODEL"], input=texts[start:start + 100])
    rows += [item.embedding for item in response.data]
    print(f"chunks {start} to {start + len(response.data) - 1}: {response.usage.prompt_tokens} tokens")

D = np.array(rows)
assert len(D) == len(chunk_ids), "every chunk needs exactly one row"
print("D:", D.shape)

chunks 0 to 99: 7668 tokens


chunks 100 to 199: 8163 tokens


chunks 200 to 271: 5708 tokens
D: (272, 1536)


Three requests, 21,539 tokens, and a matrix `D` with one row per chunk. The API returns the vectors in the order
of the texts, so `rows` follows `chunk_ids`, and the `assert` stops the notebook if the two counts ever differ.

⚠️ Keep the two lists together. IDs gathered from a set or a dictionary, or chunks embedded in another order, still give
a matrix of the right shape and plausible results, with the wrong ID on rows and no error.

### Checkpoint D

| You should see | Our run |
|---|---|
| Batches | chunks 0 to 99 (7,668 tokens), 100 to 199 (8,163), 200 to 271 (5,708) |
| Matrix | `D: (272, 1536)` |

---

## Part E: Rank the corpus for 30 questions and count hit@3

Part E measures the search: for 30 questions whose source you know, how often does the right paragraph come back?

`data/development-questions.json` holds 30 of SQuAD's own questions, from five of the articles. Each names, in `known_support`, the paragraph its answer was taken from. The sample is small so the labs stay cheap, so its scores cannot be compared with results published on the full benchmark. Embed all 30 in one call, then score them against every chunk.

In [13]:
questions = json.loads((DATA / "development-questions.json").read_text(encoding="utf-8"))
print(questions[0]["question"], "->", questions[0]["known_support"])

response = client.embeddings.create(model=settings["EMBED_MODEL"], input=[q["question"] for q in questions])
Q = np.array([item.embedding for item in response.data])
scores = Q @ D.T
print("Q:", Q.shape, "  scores:", scores.shape)

Which NFL team represented the AFC at Super Bowl 50? -> ['S00-P00']


Q: (30, 1536)   scores: (30, 272)


`Q @ D.T` (`.T` turns `D` on its side so the shapes line up) scores every question against every chunk: one row per
question, one column per chunk, 8,160 cosines from one line. Here is the row for Part A's Tesla question and its three
best chunks. `kind="stable"` keeps tied scores in chunk order, so a tie always ranks the same way.

In [14]:
tesla = next(i for i, q in enumerate(questions) if "Tesla born" in q["question"])
row = scores[tesla]
for i in np.argsort(-row, kind="stable")[:3]:
    print(f"{row[i]:.4f}  {chunk_ids[i]}")

0.7186  S03-P00-C00
0.5433  S03-P01-C00
0.4633  S03-P02-C00


First is `S03-P00-C00` (about 0.72), the start of the Tesla paragraph that gives his birth date. The three best chunks are
the question's **top 3**, in general its **top-k**.

**Known-source hit@3** turns 30 rankings into one number: a question is a hit when any of its top 3 chunks comes from a
paragraph in its `known_support`. It says that the right paragraph came back, not that the answer will be right. The
labels name only the paragraph each question was written from, so another paragraph that also answers counts as a miss.

In [15]:
def misses_at(k):
    """The questions none of whose top k chunks comes from their known_support, with those top k."""
    misses = []
    for question, row in zip(questions, scores):
        top = np.argsort(-row, kind="stable")[:k]
        if not any(chunks[i]["source_id"] in question["known_support"] for i in top):
            misses.append((question, top))
    return misses

misses_3 = misses_at(3)
print(f"known-source hit@3: {len(questions) - len(misses_3)}/{len(questions)}")
print("misses:", [question["question"] for question, _ in misses_3] or "none")
print(f"\n{settings['EMBED_MODEL']} at {embed_url(settings)}")

known-source hit@3: 30/30
misses: none

text-embedding-3-small at https://api.openai.com/v1


30 of 30, and no misses. That is a good start, not proof of quality: the questions come from only five articles, and hit@3 says nothing about whether the answer step will use the right chunk. The last line is the stamp: a hit@3 means little without the model and endpoint that produced it.

Make the test stricter. The same function with `k=1` counts **hit@1**: is the right paragraph first?

In [16]:
misses_1 = misses_at(1)
for question, top in misses_1:
    print(question["question"], "| known support:", question["known_support"], "| first chunk:", chunk_ids[top[0]])
print(f"known-source hit@1: {len(questions) - len(misses_1)}/{len(questions)}")

Who was the Super Bowl 50 MVP? | known support: ['S00-P02'] | first chunk: S00-P00-C00
known-source hit@1: 29/30


29 of 30. The one miss is *Who was the Super Bowl 50 MVP?*: its paragraph is `S00-P02`, and the chunk that came first is `S00-P00-C00`. Read the chunk that came first and the chunk that holds the answer before deciding why.

In [17]:
for chunk_id in ("S00-P00-C00", "S00-P02-C00"):
    print(chunk_id, "|", chunks[chunk_ids.index(chunk_id)]["text"], "\n")

S00-P00-C00 | Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the 

S00-P02-C00 | The Broncos took an early lead in Super Bowl 50 and never trailed. Newton was limited by Denver's defense, which sacked him seven times and forced him into three turnovers, including a fumble which they recovered for a touchdown. Denver linebacker Von Miller was named Super Bowl MVP, recording five solo tackles, 2½ sacks, and two forced fumbles. 



The first chunk is about Super Bowl 50 as a whole: the teams, the result, the date and the place. The answering chunk is about how the game went, and names the MVP only in its last sentence. A chunk's vector stands for the whole chunk, so the chunk most about the question's topic can come before the chunk with the answer. At k=3 the answering chunk is back in the list, one reason a RAG system passes several chunks to the answer step.

### Checkpoint E

| You should see | Our run |
|---|---|
| Shapes | `Q: (30, 1536)`, `scores: (30, 272)` |
| Tesla top 3 | `S03-P00-C00` about 0.72, `S03-P01-C00` 0.54, `S03-P02-C00` 0.46 |
| hit@3 and hit@1 | hit@3 30/30, no misses; hit@1 29/30, missing *Who was the Super Bowl 50 MVP?* (`S00-P00-C00` first) |

---

## Part F: Test whether a score cutoff separates answerable questions (self-study)

Part F asks what a score can tell you. A RAG system should say *the documents do not answer this* when none does; can
the top score make that call?

The Atlas question in Part C and the Tesla question in Part E both have answers, and both top scores were high
(about 0.70 and 0.72). A tempting rule follows: when the top score is below 0.60, say the corpus does not cover the
question. Try it on a question no paragraph answers: the Super Bowl 50 paragraphs never mention the national anthem.

In [18]:
cutoff = 0.60
anthem = "Who sang the national anthem at Super Bowl 50?"
response = client.embeddings.create(model=settings["EMBED_MODEL"], input=[anthem])
row = D @ np.array(response.data[0].embedding)
for i in np.argsort(-row, kind="stable")[:3]:
    print(f"{row[i]:.4f}  {chunk_ids[i]}  {chunks[i]['text'][:70]}...")
print("top score at or above the cutoff:", row.max() >= cutoff)

0.6556  S00-P00-C00  Super Bowl 50 was an American football game to determine the champion ...
0.5023  S00-P02-C00  The Broncos took an early lead in Super Bowl 50 and never trailed. New...
0.4358  S00-P01-C00  The Panthers finished the regular season with a 15–1 record, and quart...
top score at or above the cutoff: True


The top score is about 0.66, above the cutoff, so the rule calls the question covered and hands the answer step
three chunks about the right game with no answer in them. Two good examples cannot choose a cutoff; you need questions
of both kinds whose status you know.

Here are ten, labelled **before** any search by reading `data/squad-passages.json`: five that a paragraph answers, with
its ID, and five that none does (`None`), from close to the corpus to far from it. Nairobi, for example, is in the
Kenya paragraph, but only the country's population is. Predict which unanswerable question scores highest.

In [19]:
LABELLED = [
    ("What is the capital of Kenya?", "S37-P00"),
    ("What share of Earth's atmosphere is oxygen?", "S12-P00"),
    ("In what year did Genghis Khan die?", "S25-P01"),
    ("What killed the crew of Apollo 1?", "S14-P02"),
    ("What are ctenophores commonly called?", "S17-P00"),
    ("Who sang the national anthem at Super Bowl 50?", None),
    ("What is the boiling point of oxygen?", None),
    ("Who was the first person to walk on the Moon?", None),
    ("What is the population of Nairobi?", None),
    ("What will the weather be in Lisbon tomorrow?", None),
]

Embed all ten in one call, keep each question's top score and the paragraph of its top chunk, and print them from the
highest score down.

In [20]:
response = client.embeddings.create(model=settings["EMBED_MODEL"], input=[q for q, _ in LABELLED])
labelled_scores = np.array([item.embedding for item in response.data]) @ D.T

results = []
for (question, source), row in zip(LABELLED, labelled_scores):
    best = int(np.argmax(row))
    results.append((float(row[best]), source, chunks[best]["source_id"], question))

for score, source, top_source, question in sorted(results, key=lambda r: -r[0]):
    label = f"answerable ({source})" if source else "NOT answerable"
    print(f"{score:.4f}  {label:24} top chunk from {top_source}  | {question}")
print(f"\n{settings['EMBED_MODEL']} at {embed_url(settings)}")

0.6934  answerable (S17-P00)     top chunk from S17-P01  | What are ctenophores commonly called?
0.6864  answerable (S25-P01)     top chunk from S25-P01  | In what year did Genghis Khan die?
0.6556  NOT answerable           top chunk from S00-P00  | Who sang the national anthem at Super Bowl 50?
0.6516  answerable (S37-P00)     top chunk from S37-P00  | What is the capital of Kenya?
0.6234  answerable (S12-P00)     top chunk from S12-P00  | What share of Earth's atmosphere is oxygen?
0.5701  answerable (S14-P02)     top chunk from S14-P02  | What killed the crew of Apollo 1?
0.5483  NOT answerable           top chunk from S14-P00  | Who was the first person to walk on the Moon?
0.5010  NOT answerable           top chunk from S37-P00  | What is the population of Nairobi?
0.4337  NOT answerable           top chunk from S12-P00  | What is the boiling point of oxygen?
0.2227  NOT answerable           top chunk from S37-P01  | What will the weather be in Lisbon tomorrow?

text-embedding-3-s

Read from the top. The anthem question (about 0.66) scores above three of the five answerable questions: its top chunk is from the Super Bowl 50 paragraph, on the topic and silent about an anthem. *Who was the first person to walk on the Moon?* (about 0.55) sits just under *What killed the crew of Apollo 1?* (0.57), and Nairobi's population (0.50) lands on the Kenya paragraph. Only the weather in Lisbon, far from every article, scores low. And the best score of the ten, about 0.69 for the ctenophore question, comes from a chunk of `S17-P01`, not from `S17-P00`, the paragraph that says *comb jellies*.

Now apply the 0.60 rule to all ten: a question is answered when its top score is at or above the cutoff, and
refused when it is below.

In [21]:
cutoff = 0.60      # the rule from the anthem question
answerable = [score for score, source, _, _ in results if source]
unanswerable = [score for score, source, _, _ in results if not source]

print(f"cutoff {cutoff:.2f}          answered  refused")
print(f"answerable (5)       {sum(s >= cutoff for s in answerable)}         {sum(s < cutoff for s in answerable)}")
print(f"not answerable (5)   {sum(s >= cutoff for s in unanswerable)}         {sum(s < cutoff for s in unanswerable)}")

cutoff 0.60          answered  refused
answerable (5)       4         1
not answerable (5)   1         4


In our run the rule got 8 of the 10 right, and its two mistakes are the two kinds a cutoff can make: it answered the
anthem question, which no paragraph answers, and refused *What killed the crew of Apollo 1?* (about 0.57), which one
does. Raise the cutoff and it refuses more of both kinds; lower it and it answers more of both.

Try four cutoffs, on the ten and on the 30 development questions from Part E, which are all answerable.

In [22]:
development = scores.max(axis=1)                # each development question's top score, from Part E
print(f"lowest answerable {min(answerable):.4f} | highest unanswerable {max(unanswerable):.4f}\n")
print("cutoff  answerable refused  unanswerable answered  development refused")
for cut in (0.45, 0.55, 0.60, 0.66):
    print(f"{cut:.2f}    {sum(s < cut for s in answerable)} of 5               {sum(s >= cut for s in unanswerable)} of 5"
          f"                  {int((development < cut).sum())} of 30")

lowest answerable 0.5701 | highest unanswerable 0.6556

cutoff  answerable refused  unanswerable answered  development refused
0.45    0 of 5               3 of 5                  1 of 30
0.55    0 of 5               1 of 5                  3 of 30
0.60    1 of 5               1 of 5                  6 of 30
0.66    3 of 5               0 of 5                  14 of 30


No cutoff keeps every answerable question and refuses every unanswerable one: the highest unanswerable score (about 0.66) is above the lowest answerable one (about 0.57). In our run, 0.55 made the fewest mistakes on the ten, one (the anthem question is answered), yet it refused 3 of the 30 development questions, all of which the corpus answers. At 0.66 every unanswerable question was refused, and so were 3 of the 5 answerable ones and 14 of the 30 development questions.

A cosine score measures how close a chunk is to the question in topic and wording, for this model on this corpus. It is
not evidence that the chunk answers, and not a probability that it does. So the decision moves to the answer step: the
chat model reads the chunks, is told to answer only from them and to say when they do not, and code checks its reply
(Lab 02). A cutoff can still drop plainly unrelated chunks, if it is chosen on labelled questions and tested on others,
as the challenge does.

### Checkpoint F

| You should see | Our run |
|---|---|
| Anthem question | top score about 0.66, above the 0.60 cutoff |
| The 0.60 rule on the ten | answerable: 4 answered, 1 refused (Apollo 1); not answerable: 1 answered (the anthem), 4 refused |
| Ten labelled questions | lowest answerable about 0.57 (Apollo 1), highest unanswerable about 0.66 (the anthem); the ctenophore question's top chunk from `S17-P01` |
| Cutoffs | 0.55: answerable refused 0 of 5, unanswerable passed 1 of 5, development refused 3 of 30; 0.66: 3 of 5, 0 of 5 and 14 of 30 |

---

## Part G: Keep the search in `shared/search.py` (self-study)

Part G moves what you built into one module, so the labs after this one can search the corpus in one line.

You wrote the batch loop, the matrix product and the hit@3 loop by hand, and the labs that follow need all three.
`shared/search.py` holds them as four functions and adds a **cache**: the first call embeds the chunks and saves them in
a file under `.cache/` (git ignores the folder), and later calls read the file instead of calling the API. The file name
holds a hash of the endpoint, the model and every chunk text, so a changed setting or chunk never reuses old vectors. Read the two functions you will call most:

In [23]:
from shared.search import hit_at_k, search

print(inspect.getsource(search))
print(inspect.getsource(hit_at_k))

def search(question: str, chunks: list[dict], vectors: np.ndarray, settings: dict, k: int = 3) -> list:
    """The k chunks closest to the question, best first, as (score, chunk) pairs."""
    scores = vectors @ embed([question], settings)[0]
    return [(float(scores[i]), chunks[i]) for i in np.argsort(-scores, kind="stable")[:k]]

def hit_at_k(questions: list[dict], chunks: list[dict], vectors: np.ndarray, settings: dict, k: int = 3) -> list:
    """For each question, True when one of its top k chunks comes from a paragraph in its known_support."""
    scores = embed([question["question"] for question in questions], settings) @ vectors.T
    hits = []
    for question, row in zip(questions, scores):
        top = np.argsort(-row, kind="stable")[:k]
        hits.append(any(chunks[i]["source_id"] in question["known_support"] for i in top))
    return hits



`search()` is Part C's product and sort for one question, and `hit_at_k()` is Part E's loop. Both embed the questions with `embed()`, Part D's batch loop. `chunk_vectors()` embeds the chunks the same way and keeps them in the cache. Use them, and compare with what you built:

In [24]:
from shared.search import chunk_vectors, hit_at_k, search

vectors = chunk_vectors(chunks, settings)
print("vectors:", vectors.shape, "| largest difference from D:", f"{np.abs(vectors - D).max():.6f}")
print("hit@3:", sum(hit_at_k(questions, chunks, vectors, settings)), "of", len(questions))
for score, chunk in search(questions[tesla]["question"], chunks, vectors, settings):
    print(f"{score:.4f}  {chunk['chunk_id']}")
print("cache files:", sorted(path.name for path in (ROOT / ".cache").glob("chunks-*.npz")))

vectors: (272, 1536) | largest difference from D: 0.002655


hit@3: 30 of 30


0.7190  S03-P00-C00
0.5434  S03-P01-C00
0.4642  S03-P02-C00
cache files: ['chunks-text-embedding-3-small-60abc1c8bc98.npz']


The cache was empty, so the module embedded the chunks again, and single numbers differ a little from Part D's (the cell prints the largest difference): embedding calls are close to deterministic, not exactly. Scores can move in the third decimal; the order and hit@3 do not (across our runs no question's top 3 changed order). That is the cache's second purpose: the labs after this one rank with the same saved vectors, so their numbers can be compared. Run the cell again and it reads the file: no new call, the same numbers.

From here on the labs call `chunk_vectors()`, `search()` and `hit_at_k()` by name.

### Checkpoint G

| You should see | Our run |
|---|---|
| Module results | `vectors: (272, 1536)` within a few thousandths of `D`; hit@3 30 of 30; Tesla's top 3 in Part E's order |

---

## What you can do now

- Build an embeddings client by hand, embed a text and read its vector and token count, then get the same client
  from `embed_client()`.
- Compute cosine similarity, and say why a dot product is enough for unit-length vectors.
- Embed a corpus in batches, with every row tied to its chunk ID.
- Rank every chunk for many questions with one matrix product, and count known-source hit@3 and hit@1.
- Show, with questions labelled before the search, that no score cutoff separates answerable questions from
  unanswerable ones.
- Search the corpus and count hit@k with `shared/search.py`.

## Going further: shorter vectors

`text-embedding-3` models can return shorter vectors: the `dimensions` parameter keeps the first numbers and normalises
them again ([embeddings guide](https://developers.openai.com/api/docs/guides/embeddings)). A vector of 256 numbers takes a sixth of the memory of one with 1,536,
which matters at millions of chunks. Other endpoints may reject the parameter, so it is a switch in `models.env`,
`EMBED_DIMENSIONS`, empty by default. Try 256 in a copy of `settings` only:

In [25]:
short = {**settings, "EMBED_DIMENSIONS": "256"}
vectors_256 = chunk_vectors(chunks, short)
print("vectors:", vectors_256.shape, f"| {vectors_256.nbytes:,} bytes against {vectors.nbytes:,}")
print("hit@3:", sum(hit_at_k(questions, chunks, vectors_256, short)), "of", len(questions))
print("hit@1:", sum(hit_at_k(questions, chunks, vectors_256, short, k=1)), "of", len(questions))

vectors: (272, 256) | 557,056 bytes against 3,342,336


hit@3: 30 of 30


hit@1: 27 of 30


At 256 numbers the matrix takes a sixth of the memory (557,056 bytes against 3,342,336) and hit@3 stays at 30 of 30, but hit@1 falls from 29 to 27. Thirty questions from five articles are an easy test, so measure on your own questions before you shorten
vectors for a real system. To use 256 in every lab, put `EMBED_DIMENSIONS=256` in your `.env`; the cache keeps the two
sets of vectors apart.

## Challenge

Use only what this lab showed. A worked solution is in `solved/` for the instructor. Task 1 is the challenge; task 2 is
optional. For each task, keep the printed results and your answer to **Explain**.

1. **Test a cutoff on questions it was not chosen from.** In Part F, 0.55 made the fewest mistakes on the ten
   labelled questions. Does it hold on new ones?

   - In `NEW` under **Task 1**, write four questions about the corpus before you run anything: two that a paragraph
     answers, each with that paragraph's `source_id` (read `data/squad-passages.json` to find it), and two that no
     paragraph answers, with `None`. Use articles other than the ones in Part F.
   - Run the cell. It searches each question with `search()` and answers or refuses by the cutoff in `CUTOFF`.

   **Check:** four lines, each with the top score, the decision, the label and `right` or `MISTAKE`, and a count of
   mistakes. Our worked solution made no mistake on its four questions at 0.55. An unanswerable question close to an
   article's topic, like the anthem question, is the kind most likely to become one.

   **Explain:** did the cutoff make the same kinds of mistakes as on the ten? What would you need before you trusted a
   cutoff in a real system?

2. **Optional: reword the Atlas question.** In Part C, *How long does an Atlas reset link remain valid?* ranked H02,
   the answering policy, first. Does a customer's own wording keep it there?

   - In `REWORDINGS` under **Task 2**, write three questions that ask the same thing in other words, the way a customer
     might type them. Add nothing the question did not ask, such as a number of minutes.
   - Predict where H02 lands for each, then run the cell.

   **Check:** the first line is Part C's question, with H02 at rank 1 and the order H02 H04 H01 H05 H03; your three lines each
   print H02's rank and the full order. In our worked solution one of three rewordings put H02 third, behind H01 and the
   Boreal H04.

   **Explain:** which words moved H02 down, if any did? How many policies would a search have to return for every
   rewording to include H02, and what does a larger k cost the answer step?

**Task 1.** Your four questions, written before you run the cell:

In [26]:
CUTOFF = 0.55      # the Part F cutoff with the fewest mistakes on the ten labelled questions
NEW = [
    # TODO: two answerable questions as ("question", "S##-P##") and two unanswerable ones as ("question", None)
]

mistakes = 0
for question, source in NEW:
    score, chunk = search(question, chunks, vectors, settings, k=1)[0]
    decision = "answer" if score >= CUTOFF else "refuse"
    right = (decision == "answer") == (source is not None)
    mistakes += not right
    label = "answerable" if source else "NOT answerable"
    print(f"{score:.4f}  {decision:6}  {label:14}  top {chunk['source_id']}  {'right' if right else 'MISTAKE'}  | {question}")
print(f"mistakes: {mistakes} of {len(NEW)}" if NEW else "⚠️ Write your four questions in NEW, then run the cell again.")

⚠️ Write your four questions in NEW, then run the cell again.


**Task 2 (optional).** Part C's question first, then your three rewordings:

In [27]:
REWORDINGS = [
    atlas_question,          # Part C's question
    # TODO: three rewordings with the same meaning, one string each
]

for question in REWORDINGS:
    response = client.embeddings.create(model=settings["EMBED_MODEL"], input=[question])
    policy_scores = policy_vectors @ np.array(response.data[0].embedding)
    ranking = [policies[i]["source_id"] for i in np.argsort(-policy_scores)]
    print(f"H02 at rank {ranking.index('H02') + 1} | order {' '.join(ranking)} | {question}")
if len(REWORDINGS) < 4:
    print("⚠️ Add three rewordings to REWORDINGS, then run the cell again.")

H02 at rank 1 | order H02 H04 H01 H05 H03 | How long does an Atlas reset link remain valid?
⚠️ Add three rewordings to REWORDINGS, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `KeyError: 'OPENAI_API_KEY'` in Part A | Config found no key | Add it as Lab 00's README shows for your route, then run Config again |
| `ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, ... (size 256 is different from 1536)` | a question embedded with another model or another `EMBED_DIMENSIONS` than the chunks | Embed questions and chunks with the same settings; `search()` and `hit_at_k()` do it for you |
| `BadRequestError: Error code: 400 ... Invalid 'input': array length must be 2048 or less.` | more than 2,048 texts in one request | Send batches, as Part D does |
| An answerable question ranks the wrong chunk first, with no error | rows and IDs out of step (Part D) | Build the texts and the IDs from the same list, and keep the `assert` |

## Resources

- [OpenAI: embeddings guide](https://developers.openai.com/api/docs/guides/embeddings): the models, vector lengths, normalisation and the `dimensions` parameter
- [OpenAI: create embeddings](https://developers.openai.com/api/reference/resources/embeddings/methods/create): the request, its limits and the `usage` field
- [NumPy: `argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html): sorting order and `kind="stable"`
- [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard): how embedding models compare on public tasks;
  test a candidate on your own questions, as this lab does, before you switch

Next: [Lab 02](../02_Cited_Answers/README.md), *How to generate an answer that cites its sources*, or the self-study [Lab 03](../03_Chunking_With_Lineage/README.md), *How to split documents
into chunks that keep their facts*. Back to [the lab list](../README.md).